# Building a Doubly Controlled Modular Multiplier
To build  Shor's Factorizaiton Algorithm, we need the following unitary operator:
$$
U |c\rangle_1 |y\rangle_n  = \begin{cases} 
|c\rangle |ay \pmod N\rangle  & \text{if } c=1 \text{ and } y < N \\
|c\rangle |y\rangle & \text{otherwise}
\end{cases}
$$
for given $a,N\in \mathbb{Z}_+$ such that $a<N$, $gcd(a,N)=1$ and $n=\lceil \log_{2}(N) \rceil$.


In [10]:
import math
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.synthesis.qft import synth_qft_full

## Fourier Domain Phase Additions
We define the following phase addition operations. In the Fourier domain, adding a constant $a$ corresponds to applying a state dependent phase shift. The unitary transformation performs the following mapping:

$$
|x\rangle \rightarrow \exp\left( \frac{2\pi i a x}{2^{n+1}} \right) |x\rangle
$$
Because our modular operations will eventually be conditioned on multiple control qubits, we define both single-controlled and doubly-controlled versions of these phase gates.

In [11]:
def phase_add_gate(val, n_qubits, sign=1, name="PhaseAdd"):
  
    qc = QuantumCircuit(n_qubits, name=f"{name}_{val}")
    for j in range(n_qubits):
        phase = sign * 2 * np.pi * val * (2**j) / (2**n_qubits)
        qc.p(phase, j)
    return qc.to_gate()

def ctrl_phase_add_gate(val, n_targets, sign=1):

    gate = phase_add_gate(val, n_targets, sign)
    return gate.control(1)

def mcp_phase_add_gate(val, n_targets, sign=1):

    gate = phase_add_gate(val, n_targets, sign)
    return gate.control(2)

## The Core Multiplier Block

Our goal is to construct a unitary gate that performs controlled modular multiplication into an accumulator register $x$, applying the transformation:
$$
|y\rangle |x\rangle \rightarrow |y\rangle |(x + a \cdot y) \pmod N\rangle
$$

### Step A: Draper's QFT Addition
We start with **Draper's QFT addition algorithm**. On a standard $n$-qubit register, Draper's algorithm applies the Quantum Fourier Transform over $2^{n}$, rotates the phases by a constant $a$, and applies the Inverse QFT to output:
$$
|x\rangle \rightarrow |(x + a) \pmod{2^n}\rangle
$$

### Step B: Upgrading to Modulo $N$
Because we defined $n = \lceil \log_2(N) \rceil$, the sum $x+a$ can exceed $N$. To figure out $(x+a) \pmod N$, we can subtract $N$ from the sum and simply check if the result is negative. 

In order to do this, we add an **extra qubit**, expanding our $x$ register to $n+1$ qubits, and perform our Fourier transforms over $2^{n+1}$. The most significant bit of this expanded register (The rightmost bit) will be $1$ **if and only if** there is an underflow (meaning $x+a < N$). 


We copy this MSB to a separate ancilla qubit using a CNOT gate. If the ancilla is $1$, we know an underflow occurred, and we conditionally add $N$ back to restore the correct modulo target. 

### Step C: Scaling to Multiplication
To construct a multiplier from our adder, we observe the following relation:

$$
ay \pmod N = \sum y_i (a 2^i \pmod N) \pmod N
$$

This demonstrates that modular multiplication is simply a series of modular additions of the shifted constants $(a 2^i \pmod N)$, where each addition is controlled by the corresponding $y_i$ qubit.

### Step D: Restoring the Ancilla Qubit

Ancilla qubit needs to be reset to $|0\rangle$ at each step for the future use. The following fact implies that it suffices to subtract $a_i = a 2^i \pmod N$), and check the sign of the resulting number, which is essentially the same with the Step B. 

$$
(x + a_i) \pmod N \ge a_i \iff x + a_i < N
$$
.

In [12]:
def append_ctrl_mod_mult(qc, a_val, N, c_qubit, y_reg, x_reg, ancilla_qubit):
    n, x_len = len(y_reg), len(x_reg)
    qft = synth_qft_full(x_len, do_swaps=True)
    iqft = qft.inverse()

    for i in range(n):
        a_shifted = (a_val * (2**i)) % N 
        if a_shifted == 0: continue
            
        # Step 1: Add a_shifted
        qc.compose(qft, qubits=x_reg, inplace=True)
        qc.append(mcp_phase_add_gate(a_shifted, x_len), [c_qubit, y_reg[i]] + list(x_reg))
        qc.append(phase_add_gate(N, x_len, sign=-1), x_reg)
        qc.compose(iqft, qubits=x_reg, inplace=True)
        
        # Step 2: Overflow bit logic
        qc.cx(x_reg[-1], ancilla_qubit)
        
        # Step 3: Conditional subtract
        qc.compose(qft, qubits=x_reg, inplace=True)
        qc.append(ctrl_phase_add_gate(N, x_len, sign=1), [ancilla_qubit] + list(x_reg))
        qc.append(mcp_phase_add_gate(a_shifted, x_len, sign=-1), [c_qubit, y_reg[i]] + list(x_reg))
        qc.compose(iqft, qubits=x_reg, inplace=True)
        
        # Step 4: Reset ancilla
        qc.x(x_reg[-1])
        qc.cx(x_reg[-1], ancilla_qubit)
        qc.x(x_reg[-1])
        
        # Step 5: Final Add
        qc.compose(qft, qubits=x_reg, inplace=True)
        qc.append(mcp_phase_add_gate(a_shifted, x_len), [c_qubit, y_reg[i]] + list(x_reg))
        qc.compose(iqft, qubits=x_reg, inplace=True)

##  The Controlled Multiplier

To finalize our modular multiplier, we must add  control conditions and convert the operation to run entirely on the $y$ register.

### Step A:  Control Qubits
We want the gate to execute only if two conditions are met:
1. The control qubit $c$ is $|1\rangle$.
2. Our input register $y$ is strictly less than $N$. 

To verify $y < N$, we temporarily use the empty $x$ register to compute $y - N$. Just like in our addition block, we check the most significant bit (MSB). If the MSB is $1$, we know $y - N < 0$, which guarantees $y < N$. We copy this MSB to a flag qubit, cleanly uncompute the $x$ register back to $|0\rangle$, and use a Toffoli (CCX) gate to combine our external control $c$ and this flag qubit into a single **effective control** qubit.

### Step B: Constructing the Final Unitary Mapping
Our core gate currently takes inputs $|y\rangle |x\rangle$ and outputs $|y\rangle |(x + ay) \pmod N\rangle$. Assuming the $x$ register is initialized to $|0\rangle$, this yields $|y\rangle |ay \pmod N\rangle$. 

To turn this into a gate that maps $|y\rangle \rightarrow |ay \pmod N\rangle$ while leaving $x$ empty, we use a three-part trick:

1. **Forward Multiplier:** We apply our core multiplier block to generate the state $|y\rangle |ay \pmod N\rangle$.
2. **Register Swap:** We swap $y$ and $x$ registers. Our state is now $|ay \pmod N\rangle |y\rangle$. 
3. **Inverse Multiplication:** To clear the $x$ register (which now holds the original $y$) back to $|0\rangle$, we run the core multiplier block with $N-a^{-1}$. The existence of the inverse is guaranteed by $gcd(a,N)=1$.


By clearing the $x$ register, our final transformation becomes the desired unitary transformation:

$$
|c\rangle |y\rangle |0\rangle \rightarrow |c\rangle |ay \pmod N\rangle |0\rangle
$$

In [13]:
def build_conditional_multiplier(a: int, N: int):
    n = math.ceil(math.log2(N))
    x_len = n + 1 
    
    # Register Setup
    c_reg = QuantumRegister(1, name='c')
    y_reg = QuantumRegister(n, name='y')
    x_reg = QuantumRegister(x_len, name='x')
    ancilla = QuantumRegister(1, name='ancilla')
    flag = QuantumRegister(1, name='flag')
    c_eff = QuantumRegister(1, name='c_eff')
    
    qc = QuantumCircuit(c_reg, y_reg, x_reg, ancilla, flag, c_eff, name=f"ModMult_{a}")
    
    qft = synth_qft_full(x_len, do_swaps=True)
    iqft = qft.inverse()

    # --- 1. Compute 'y < N'  ---
    qc.compose(qft, qubits=x_reg, inplace=True)
    for i in range(n):
        qc.append(ctrl_phase_add_gate(2**i, x_len), [y_reg[i]] + list(x_reg))
    qc.append(phase_add_gate(N, x_len, sign=-1), x_reg)
    qc.compose(iqft, qubits=x_reg, inplace=True)
    
    qc.cx(x_reg[-1], flag[0])
    
    # --- 2. Uncompute 'y < N' to clean x_reg ---
    qc.compose(qft, qubits=x_reg, inplace=True)
    qc.append(phase_add_gate(N, x_len, sign=1), x_reg)
    for i in range(n):
        qc.append(ctrl_phase_add_gate(2**i, x_len, sign=-1), [y_reg[i]] + list(x_reg))
    qc.compose(iqft, qubits=x_reg, inplace=True)

    # --- 3. Conditional Logic ---
    qc.ccx(c_reg[0], flag[0], c_eff[0])
    
    # Forward Multiplier
    append_ctrl_mod_mult(qc, a, N, c_eff[0], y_reg, x_reg, ancilla[0])
    
    # Swap y and x (result is in x, we want it back in y)
    for i in range(n):
        qc.cswap(c_eff[0], y_reg[i], x_reg[i])
        
    # Reverse Multiplier (to clear x_reg using a_inverse)
    a_inv_neg = (N - pow(a, -1, N)) % N 
    append_ctrl_mod_mult(qc, a_inv_neg, N, c_eff[0], y_reg, x_reg, ancilla[0])
    
    return qc

## Testing
We construct a conditional multiplier circuit for $N=15$ and $a=4$. To test the circuit with an input like $y=6$, we initialize a quantum register of the appropriate size and apply $X$ gates to set the control qubit to $1$ and encode the value $6$ into the $y$ register.

In [33]:
a = 4
N = 15
y_val = 6  # The input value you want to multiply
c_val = 1  # Control bit (1 to activate, 0 to bypass)

# Build the Multiplier Block 
# (Assuming your main function is named build_conditional_multiplier)
multiplier_circuit = build_conditional_multiplier(a, N)
total_qubits = multiplier_circuit.num_qubits
n = math.ceil(math.log2(N))

# Set Up the test register
test_qc = QuantumCircuit(total_qubits)

# Initialize the control qubit (index 0)
if c_val == 1:
    test_qc.x(0)

# Initialize the y register (starts at index 1)
# Qiskit uses little-endian ordering, so we reverse the binary string
y_bin = bin(y_val)[2:] 
y_indices = [i + 1 for i, bit in enumerate(reversed(y_bin)) if bit == '1']

if y_indices:
    test_qc.x(y_indices)


# Apply the multiplier

test_qc.append(multiplier_circuit, range(total_qubits))

#  Read the Results
# Measure the probabilities of the y register (qubits 1 to n)
probs = Statevector(test_qc).probabilities_dict(range(1, n + 1), decimals=3)

# Filter out zero-probability states and convert binary keys to decimal
results = {int(k, 2): float(v) for k, v in probs.items() if v > 0}

print(f"Test Parameters: c={c_val}, y={y_val}, a={a}, N={N}")
print(f"Expected Math: ({a} * {y_val}) mod {N} = {(a * y_val) % N}")
print(f"Circuit Output (Decimal: Probability): {results}")



Test Parameters: c=1, y=6, a=4, N=15
Expected Math: (4 * 6) mod 15 = 9
Circuit Output (Decimal: Probability): {9: 1.0}
